# Lec-6: Language (Natural Language Processing)
---


**Syntax:** _Structure of the sentences._  
- "Just before nine o'clock Sherlock Holmes stepped briskly into the room" << Syntactically valid.
- "Before just Sherlock stepped into briskly Holmes nine o'clock the room" << Syntactically not valid.   
- "I saw the man on the mountain with a telescope" >> Ambiguous. I with a telescope ? or the man with a telescope ?

**Semantics:** _Meaning of the sentences._   
- "Colorless green ideas sleep furiously." << Whoever said this was drunk !

**Formal Grammar:**   
- A system of rules for generating sentences.
- **Context Free Grammar:**
  - "She saw the city", each of this words are _Terminal Symbols_.
  - Associate these words with a _Non-Terminal Symbols_.   

    |Non-Terminal Symbols| N (noun) | V (verb) | D (determiner) | N (noun) |
    |:---|:---:|:---:|:---:|:---:|
    |Terminal Symbols| She | saw | the | city |

  - Now, we need to teach which non-terminal symbols could be replaced by which terminal symbols.
    - $N \rightarrow she|city|car|Harry|...$
    - $D \rightarrow the|a|an|...$
    - $V \rightarrow saw|ate|walked|...$
    - $P \rightarrow to|on|over|...$
    - $ADJ \rightarrow blue|busy|old|...$
  - Rule example:
    - $NP \rightarrow N \ | D\ \ N$. (A noun phrase (NP) can be just a noun (N) or a noun followed by a determiner (D N)).   
      <img src="image.png" alt="grammar_1" width="50%" height="auto"/>   
    - S -> NP VP.   
      <img src="image-1.png" alt="grammar_2" width="50%" height="auto"/>   


> Python Lib: **nltk**. (natural language toolkit).





In [ ]:
# Imports
import math, os, sys
import nltk
## Run following once to download necessary packages
#nltk.download('punkt')
#nltk.download('wordnet')
#nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/manashchakraborty/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/manashchakraborty/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/manashchakraborty/nltk_data...


In [4]:
### CFG Parsing
# nltk can be used to define context free grammar (CFG)
grammar = nltk.CFG.fromstring("""
    S -> NP VP

    NP -> D N | N
    VP -> V | V NP
    
    D -> "a" | "an" | "the"
    N -> "She" | "John" | "Mary" | "dog" | "cat" | "city" | "car" | "He"
    V -> "saw" | "ate" | "walked"
""")

parser = nltk.ChartParser(grammar)

sentence = input("Enter a sentence: ").split() # Split the input sentence into words. pivots at spaces.
count = 0
try:
    for tree in parser.parse(sentence):
        tree.pretty_print() # Print the parse tree in a readable format.
        count += 1
        print(f"Parse tree count = {count}:")

except ValueError as e:
    print("Error parsing sentence:", e)

if count == 0:
    print("No valid parse trees found.")


         S              
  _______|___            
 |           VP         
 |    _______|___        
 NP  |           NP     
 |   |        ___|___    
 N   V       D       N  
 |   |       |       |   
She saw     the     city

Parse tree count = 1:


> Better example: see `\src6\cfg\cfg1.py`.

_Note, building grammar like this will take a long time and won't be reliable and language in general is complex and possibly evolving over time.
We could instead take a more statistical approach as well_.  

### $n$-gram:   
- a contiguous sequence of $n$ items from a sample of text.
- *$tri$-grams:* _"How often have I said to you that when you have eliminated the impossible whatever remains, however improbable, must be the truth ? "_. Here trigrams are:
    - "How often have",
    - "often have I",
    - "have I said",
    - "I said to", etc.

- $n$-grams of $2$ are typically known as _Bi-gram_.

In [ ]:
### Bigrams from text corpus
from collections import Counter
#======== [Function] =================================
def load_data(directory):
    contents = []

    # Read all files and extract words
    for filename in os.listdir(directory):
        with open(os.path.join(directory, filename)) as f:
            contents.extend([
                word.lower() for word in
                nltk.word_tokenize(f.read())
                if any(c.isalpha() for c in word)
            ])
    return contents
#_____________________________________________________

corpus = load_data("src6/ngrams/holmes")
print(len(corpus), "words loaded.")

n = int(input("How many words in each n-gram? "))

# Compute n-grams
ngrams = Counter(nltk.ngrams(corpus, n))

# Print 10 most common n-grams
for ngram, freq in ngrams.most_common(10):
    print(f"{' '.join(ngram)}  :  {freq}")


178282 words loaded.
of the  :  1158
in the  :  879
it was  :  521
to the  :  498
it is  :  463
i have  :  457
that i  :  405
at the  :  378
and i  :  370
and the  :  332


> Code Explained:   
1. Tokenize the file content. `f.read()` reads the entire file as a string.
`nltk.word_tokenize(f.read())` splits the text into tokens (words and punctuation).
2. Filter and process tokens. For each word in the tokenized list:   
`if any(c.isalpha() for c in word)` keeps only tokens that contain at least one alphabetic character (filters out pure punctuation). `word.lower()` converts the word to lowercase.
3. Extend the contents list
The filtered, lowercased words are added to the contents list using contents.`extend([...])`.

**Tokenization:**. 
The task of splitting a sequence of chars into pieces (tokens).   

**Markov Chain:**
- of tokens to predict next item.



In [11]:
# TODO markov chain

## Word Classification
### Sentiment Analysis

**Bag-of-words Model:** _Order is omitted here._
- Model that represents text as an unordered collection of words.
- Use it to build **Naive Bayes Classifier**.
    - _Bayes' Rule:_ $P(b|a) = \frac{P(a|b)P(b)}{P(a)}$.
    - Interested in: $P(possitive), P(negative)$.
    - $P(\text{possitive sentiment} | \text{"my grandson loved it"})$.
    - Here is where _Bag-of-words model_ comes in. Instead of treating this as a ordered sequence of words. We will treat this a unordered collection of words.
        - $P(\text{possitive sentiment} | \text{"my", "grandson", "loved", "it"})$.

- So then we calculate following:
    - $P(\text{possitive sentiment} | \text{"my", "grandson", "loved", "it"}) = \frac{P(\text{"my", "grandson", "loved", "it" are in text}) | P(\text{possitive sentiment})}{P(\text{"my", "grandson", "loved", "it" are in text})}$
- In other words:
    - $P(\text{possitive sentiment} | \text{"my", "grandson", "loved", "it"}) \propto P(\text{"my", "grandson", "loved", "it" are in text}) | P(\text{possitive sentiment})$.
- Based on the rules of Joint probability:
    - $P(\text{possitive sentiment} | \text{"my", "grandson", "loved", "it"}) \propto P(\text{"my", "grandson", "loved", "it" are in text}, \text{possitive sentiment})$.
- Naive assumption:
    - All words are indipendent of each other. Meaning if "gradson" is in the text, doesn't change the probability of "loved" being in the text.
    - Then proportionality equation above becomes a _"naively proportional equation"_ $\approx \propto$. 
    - $P(\text{possitive sentiment} | \text{"my", "grandson", "loved", "it"}) \approx \propto P(\text{pos sentiment}) P("my"|positive) P("grandson"|possitive)P("loved"|possitive)P("it"|possitive)$.
- Based on some training data:
    - $P(\text{possitive sentiment}) = \frac{\text{number of positive samples}}{\text{number of total samples}}$.
    - $P("loved"|possitive) = \frac{\text{number of positive samples with "loved"}}{\text{number of positive samples}}$.
- Finally, we can normalize the this to finally get some probabilistic answer.


- Example:   
    <img src="image-2.png" alt="BoW-1" width="50%" height="auto"/>.  
    - Calculate the probability by multiplying corresponding values for possitive and negative sentiment:   
    <img src="image-3.png" alt="BoW-2" width="50%" height="auto"/>.  
    - Normalize the values to get pobability:   
    <img src="image-4.png" alt="BoW-3" width="50%" height="auto"/>.  

- Potential Issue:
    - What one of the probability for a word is $0$, after multiplying everything becomes zero.   
    <img src="image-5.png" alt="BoW-4" width="50%" height="auto"/>.  
    - Solution:
        - **Additive Smoothing:** Adding a value $\alpha$ to each value in our distribution to smooth the data.
        - **Laplace Smoothing:** Adding $1$ to each value in our distribution pretending we've seen each value one more time than we actually have.

In [ ]:
### Sentiment Analysis
#===== [Function] ==============================================
def extract_words(document):
    return set(
        word.lower() for word in nltk.word_tokenize(document)
        if any(c.isalpha() for c in word)
    )
#_______________________________________________________________
#===== [Function] ==============================================
def load_data(directory):
    result = []
    for filename in ["positives.txt", "negatives.txt"]:
        with open(os.path.join(directory, filename)) as f:
            result.append([
                extract_words(line)
                for line in f.read().splitlines()
            ])
    return result
#_______________________________________________________________
#===== [Function] ==============================================
def generate_features(documents, words, label):
    features = []
    for document in documents:
        features.append(({
            word: (word in document)
            for word in words
        }, label))
    return features
#_______________________________________________________________
#===== [Function] ==============================================
def classify(classifier, document, words):
    document_words = extract_words(document)
    features = {
        word: (word in document_words)
        for word in words
    }
    return classifier.prob_classify(features)
#_______________________________________________________________

positive, negative = load_data("src6/sentiment/corpus") # sequence unpacking of python lists.

# Create a set of all words
all_words = set()
for doc in positive + negative:
    all_words.update(doc)

# Extract Feature from Text
# This converts text documents into a format that NLTK's NaiveBayesClassifier can use for training and classification.
training = []
training.extend(generate_features(positive, all_words, "positive")) # Note: .extend(), It unpacks the iterable and adds each item individually.
training.extend(generate_features(negative, all_words, "negative")) # Note: .extend(), It unpacks the iterable and adds each item individually.

# Classify New Samples
classifier = nltk.NaiveBayesClassifier.train(training)
s = input("Enter a sentence: ")
result = (classify(classifier, s, all_words))
for key in result.samples():
    print(f"{key}: {result.prob(key):.4f}")




positive: 0.9946
negative: 0.0054


### How to take word and turn them into numbers
- "He wrote a book"
- Vectorize: 
    - **One-hot Representation**
        - He = $[1, 0, 0, 0]$.
        - wrote = $[0, 1, 0, 0]$.
        - a = $[0, 0, 1, 0]$.
        - book = $[0, 0, 0, 1]$.
        - Issues:
            - long sentences will generate a long vector.
            - Similar words should have similar vector, but that it not guaranteed with _One-hot Representation_.
    - **Distributed Representation:** representation of meaning distributed across multiple values.    
    <img src="image-6.png" alt="distRepresnt-1" width="50%" height="auto"/>.  
        - `word2vec` Model. Model for generating word vectors.   
        <img src="image-7.png" alt="distRepresnt-2" width="50%" height="auto"/>.  
        - This will help us analyze and compute more complex relationship between words. See following image in 2D, but in reality this concept is happening in much more higher dimension.     
        <img src="image-8.png" alt="distRepresnt-3" width="50%" height="auto"/>.  
    


    




In [1]:
# Imports
from scipy.spatial.distance import cosine
import math
import numpy as np

In [ ]:
#======[Functions]====================================
def distance(word1, word2):
    return cosine(word1, word2)

def closest_words(embedding):
    distances = {
        w: distance(embedding, words[w]) for w in words
    }
    print("Closest words: ")
    print(sorted(distances, key=lambda w: distances[w])[:10])
    return sorted(distances, key=lambda w: distances[w])[:10]

def closest_word(embedding):
    return closest_words(embedding=embedding)[0]
#_____________________________________________________

### Vectorized Words
with open("src6/vectors/words.txt", encoding="utf-8") as f:
    words = dict()
    for line in f:
        row = line.split() # space separated items are split
        word = row[0]
        vector = np.array([float(x) for x in row[1:]]) # casting to float is imp, otherwise some chars may not appear as numbers
        words[word] = vector


In [45]:
#print(f"king: {words["king"]}")
closest_word(words["king"] - words["man"] + words["woman"])

Closest words: 
['queen', 'king', 'empress', 'prince', 'duchess', 'princess', 'consort', 'monarch', 'dowager', 'throne']


'queen'

### Encoder-Decoder Architecture of (RNN)
- Sometimes it is important that we keep track of the sequence of the inputs. For example, if we input a question to NN and expect an answer from NN, note that the input question could be of different length, so the input layer can expect different sizes of input.
- In these kind of scenario, we will use RNN, where we will run NN multiple times, and each time we will keep track of a hidden state.
- In _Encoder-Decoder Architecture_ we will encode a question into a state state, and then decode it to get output.

Encoding:   
<img src="image-9.png" alt="Enc-Dec_1" width="20%" height="auto"/>   
<img src="image-10.png" alt="Enc-Dec_2" width="20%" height="auto"/>   
Decoding:   
<img src="image-11.png" alt="Enc-Dec_3" width="20%" height="auto"/>   
+ also add an `end` token to end the statement at the end.   

<img src="image-12.png" alt="Enc-Dec_4" width="40%" height="auto"/>   

**Issues of Encoder-Decoder Architecture:**   
- At the end of _Input Sequence_ we have a final encoder hidden state which is then passed along to the _Output sequence_.
- For complex sentences, one hidden state is not enough to fully contextualize the final hidden state.
- Also, at each layer the hidden state is closely related to the corresponding word. So we can say not all of the hidden state is equally important. Meaning some of these hidden states might be more important to generate an accurate output sequence.
- **Solution:**
  - _Attention_
- Another issue is RNN is difficult to parallelize. Because they generate hidden states in sequence, for long sentences it becomes compurationaly time consuming.
- **Solution:**
  - Different architecture is needed. **Transformer Architecture**.

### Attention:
- A mechanism to calculate an attention score for each word in input sequence.   
<img src="image-13.png" alt="attent_1" width="30%" height="auto"/>   

- Calculating attention score, then take sum of weighted average of the hidden states, make final hidden state:   
  <img src="image-14.png" alt="attent_2" width="30%" height="auto"/>    

### Transformer:



